In [4]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)

In [5]:
df = pd.read_csv("/content/data - data.csv", header=None)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (569, 32)
         0  1      2      3       4       5        6        7       8   \
0    842302  M  17.99  10.38  122.80  1001.0  0.11840  0.27760  0.3001   
1    842517  M  20.57  17.77  132.90  1326.0  0.08474  0.07864  0.0869   
2  84300903  M  19.69  21.25  130.00  1203.0  0.10960  0.15990  0.1974   
3  84348301  M  11.42  20.38   77.58   386.1  0.14250  0.28390  0.2414   
4  84358402  M  20.29  14.34  135.10  1297.0  0.10030  0.13280  0.1980   

        9   ...     22     23      24      25      26      27      28      29  \
0  0.14710  ...  25.38  17.33  184.60  2019.0  0.1622  0.6656  0.7119  0.2654   
1  0.07017  ...  24.99  23.41  158.80  1956.0  0.1238  0.1866  0.2416  0.1860   
2  0.12790  ...  23.57  25.53  152.50  1709.0  0.1444  0.4245  0.4504  0.2430   
3  0.10520  ...  14.91  26.50   98.87   567.7  0.2098  0.8663  0.6869  0.2575   
4  0.10430  ...  22.54  16.67  152.20  1575.0  0.1374  0.2050  0.4000  0.1625   

       30       31  
0  0.4601  0.11890  
1

In [6]:
df.describe()


,0,2,3,4,5,6,7,8,9,10,...,22,23,24,25,26,27,28,29,30,31
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [7]:
df.isna().any()

,0
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


In [8]:
df = df.drop(df.columns[0], axis=1)

print("Dataset after removing ID column:")
print(df.head())

Dataset after removing ID column:
  1      2      3       4       5        6        7       8        9       10  \
0  M  17.99  10.38  122.80  1001.0  0.11840  0.27760  0.3001  0.14710  0.2419   
1  M  20.57  17.77  132.90  1326.0  0.08474  0.07864  0.0869  0.07017  0.1812   
2  M  19.69  21.25  130.00  1203.0  0.10960  0.15990  0.1974  0.12790  0.2069   
3  M  11.42  20.38   77.58   386.1  0.14250  0.28390  0.2414  0.10520  0.2597   
4  M  20.29  14.34  135.10  1297.0  0.10030  0.13280  0.1980  0.10430  0.1809   

   ...     22     23      24      25      26      27      28      29      30  \
0  ...  25.38  17.33  184.60  2019.0  0.1622  0.6656  0.7119  0.2654  0.4601   
1  ...  24.99  23.41  158.80  1956.0  0.1238  0.1866  0.2416  0.1860  0.2750   
2  ...  23.57  25.53  152.50  1709.0  0.1444  0.4245  0.4504  0.2430  0.3613   
3  ...  14.91  26.50   98.87   567.7  0.2098  0.8663  0.6869  0.2575  0.6638   
4  ...  22.54  16.67  152.20  1575.0  0.1374  0.2050  0.4000  0.1625  0.2364   

In [9]:
x = df.drop(df.columns[0], axis=1)
y = df.iloc[:, 0]

print("Features:", x.shape)
print("Target:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Features: (569, 30)
Target: (569,)

Target distribution:
1
B    357
M    212
Name: count, dtype: int64


In [10]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [11]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Training data:", x_train_scaled.shape)
print("Testing data:", x_test_scaled.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [12]:
svm = SVC(random_state=42)

svm.fit(x_train_scaled, y_train)

print("SVM model trained successfully")

SVM model trained successfully


In [13]:
y_pred_svm = svm.predict(x_test_scaled)

print("SVM predictions:")
print(y_pred_svm[:10])

SVM predictions:
['B' 'M' 'B' 'B' 'B' 'B' 'M' 'B' 'B' 'B']


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   1       569 non-null    object 
 1   2       569 non-null    float64
 2   3       569 non-null    float64
 3   4       569 non-null    float64
 4   5       569 non-null    float64
 5   6       569 non-null    float64
 6   7       569 non-null    float64
 7   8       569 non-null    float64
 8   9       569 non-null    float64
 9   10      569 non-null    float64
 10  11      569 non-null    float64
 11  12      569 non-null    float64
 12  13      569 non-null    float64
 13  14      569 non-null    float64
 14  15      569 non-null    float64
 15  16      569 non-null    float64
 16  17      569 non-null    float64
 17  18      569 non-null    float64
 18  19      569 non-null    float64
 19  20      569 non-null    float64
 20  21      569 non-null    float64
 21  22      569 non-null    float64
 22  23

In [15]:
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm, pos_label="M")
rec_svm = recall_score(y_test, y_pred_svm, pos_label="M")
f1_svm = f1_score(y_test, y_pred_svm, pos_label="M")

print("SVM Performance Before Optimization\n")
print("Accuracy:", acc_svm)
print("Precision:", prec_svm)
print("Recall:", rec_svm)
print("F1-Score:", f1_svm)

SVM Performance Before Optimization

Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1-Score: 0.9629629629629629


In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scorer = make_scorer(f1_score, pos_label="M")

scores_svm = cross_val_score(
    svm,
    x_train_scaled,
    y_train,
    cv=cv,
    scoring=f1_scorer
)

print("SVM Cross-Validation Scores:", scores_svm)
print("Mean CV F1 score:", scores_svm.mean())

SVM Cross-Validation Scores: [0.95384615 0.97142857 0.95522388 0.95384615 0.97058824]
Mean CV F1 score: 0.9609865990024022


In [17]:
param_grid_svm = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["rbf", "linear", "poly"],
    "gamma": ["scale", "auto", 0.01, 0.1]
}

print("SVM parameter grid created")

SVM parameter grid created


In [18]:
grid_svm = GridSearchCV(
    SVC(),
    param_grid_svm,
    cv=cv,
    scoring=f1_scorer,
    n_jobs=-1
)

print("SVM Grid Search created")

SVM Grid Search created


In [19]:
grid_svm.fit(x_train_scaled, y_train)

print("SVM hyperparameter tuning completed")

SVM hyperparameter tuning completed


In [20]:
print("Best Parameters:", grid_svm.best_params_)
print("Best CV F1-Score:", grid_svm.best_score_)

Best Parameters: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV F1-Score: 0.9697549589713768


In [21]:
best_svm = grid_svm.best_estimator_

y_pred_svm_opt = best_svm.predict(x_test_scaled)

print("SVM Performance After Optimization\n")
print("Accuracy:", accuracy_score(y_test, y_pred_svm_opt))
print("Precision:", precision_score(y_test, y_pred_svm_opt, pos_label="M"))
print("Recall:", recall_score(y_test, y_pred_svm_opt, pos_label="M"))
print("F1-Score:", f1_score(y_test, y_pred_svm_opt, pos_label="M"))

SVM Performance After Optimization

Accuracy: 0.9824561403508771
Precision: 1.0
Recall: 0.9523809523809523
F1-Score: 0.975609756097561


In [22]:
df.dtypes

,0
1,object
2,float64
3,float64
4,float64
5,float64
6,float64
7,float64
8,float64
9,float64
10,float64


In [23]:
rf = RandomForestClassifier(random_state=42)

rf.fit(x_train_scaled, y_train)

print("Random Forest model trained successfully")

Random Forest model trained successfully


In [24]:
y_pred_rf = rf.predict(x_test_scaled)

print("Random Forest predictions:")
print(y_pred_rf[:10])

Random Forest predictions:
['B' 'M' 'B' 'M' 'B' 'B' 'M' 'B' 'B' 'B']


In [25]:
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, pos_label="M")
recall_rf = recall_score(y_test, y_pred_rf, pos_label="M")
f1_rf = f1_score(y_test, y_pred_rf, pos_label="M")

print("Random Forest Performance Before Optimization\n")
print("Accuracy:", acc_rf)
print("Precision:", prec_rf)
print("Recall:", recall_rf)
print("F1-Score:", f1_rf)

Random Forest Performance Before Optimization

Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1-Score: 0.9629629629629629


In [26]:
scores_rf = cross_val_score(
    rf,
    x_train_scaled,
    y_train,
    cv=cv,
    scoring=f1_scorer
)

print("Random Forest Cross-Validation F1 Scores:", scores_rf)
print("Mean CV F1-Score:", scores_rf.mean())

Random Forest Cross-Validation F1 Scores: [0.93939394 0.98550725 0.92307692 0.95652174 0.95522388]
Mean CV F1-Score: 0.9519447457150247


In [27]:
param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10]
}

print("Random Forest parameter grid created")

Random Forest parameter grid created


In [28]:
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=cv,
    scoring=f1_scorer,
    n_jobs=-1
)

print("Random Forest Grid Search created")

Random Forest Grid Search created


In [29]:
grid_rf.fit(x_train_scaled, y_train)

print("Random Forest hyperparameter tuning completed")

Random Forest hyperparameter tuning completed


In [30]:
print("Best Parameters:", grid_rf.best_params_)
print("Best CV F1-Score:", grid_rf.best_score_)

Best Parameters: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1-Score: 0.9583741372190431


In [31]:
# Retrieve the best estimator directly from the fitted grid search

best_rf = grid_rf.best_estimator_

y_pred_rf_opt = best_rf.predict(x_test_scaled)

print("Random Forest Performance After Optimization\n")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_opt))
print("Precision:", precision_score(y_test, y_pred_rf_opt, pos_label="M"))
print("Recall:", recall_score(y_test, y_pred_rf_opt, pos_label="M"))
print("F1-Score:", f1_score(y_test, y_pred_rf_opt, pos_label="M"))

Random Forest Performance After Optimization

Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1-Score: 0.9629629629629629


In [32]:
ada = AdaBoostClassifier(random_state=42)

ada.fit(x_train_scaled, y_train)

print("AdaBoost model trained successfully")

AdaBoost model trained successfully


In [33]:
y_pred_ada = ada.predict(x_test_scaled)

print("AdaBoost predictions:")
print(y_pred_ada[:10])

AdaBoost predictions:
['B' 'M' 'B' 'M' 'B' 'B' 'M' 'B' 'B' 'B']


In [34]:
acc_ada = accuracy_score(y_test, y_pred_ada)
prec_ada = precision_score(y_test, y_pred_ada, pos_label="M")
recall_ada = recall_score(y_test, y_pred_ada, pos_label="M")
f1_ada = f1_score(y_test, y_pred_ada, pos_label="M")

print("AdaBoost Performance Before Optimization\n")
print("Accuracy:", acc_ada)
print("Precision:", prec_ada)
print("Recall:", recall_ada)
print("F1-Score:", f1_ada)

AdaBoost Performance Before Optimization

Accuracy: 0.9824561403508771
Precision: 1.0
Recall: 0.9523809523809523
F1-Score: 0.975609756097561


In [35]:
scores_ada = cross_val_score(
    ada,
    x_train_scaled,
    y_train,
    cv=cv,
    scoring=f1_scorer
)

print("AdaBoost Cross-Validation F1 Scores:", scores_ada)
print("Mean CV F1-Score:", scores_ada.mean())

AdaBoost Cross-Validation F1 Scores: [0.92063492 0.97142857 0.93939394 0.95652174 0.98507463]
Mean CV F1-Score: 0.9546107594907076


In [36]:
param_grid_ada = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0]
}

print("AdaBoost parameter grid created")

AdaBoost parameter grid created


In [37]:
grid_ada = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid_ada,
    cv=cv,
    scoring=f1_scorer,
    n_jobs=-1
)

print("AdaBoost Grid Search created")

AdaBoost Grid Search created


In [38]:
grid_ada.fit(x_train_scaled, y_train)

print("AdaBoost hyperparameter tuning completed")

AdaBoost hyperparameter tuning completed


In [39]:
print("Best Parameters:", grid_ada.best_params_)
print("Best CV F1-Score:", grid_ada.best_score_)

Best Parameters: {'learning_rate': 1.0, 'n_estimators': 200}
Best CV F1-Score: 0.9609916988128233


In [40]:
best_ada = grid_ada.best_estimator_

y_pred_ada_opt = best_ada.predict(x_test_scaled)

print("AdaBoost Performance After Optimization\n")
print("Accuracy:", accuracy_score(y_test, y_pred_ada_opt))
print("Precision:", precision_score(y_test, y_pred_ada_opt, pos_label="M"))
print("Recall:", recall_score(y_test, y_pred_ada_opt, pos_label="M"))
print("F1-Score:", f1_score(y_test, y_pred_ada_opt, pos_label="M"))

AdaBoost Performance After Optimization

Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1-Score: 0.9629629629629629


In [41]:
comparison = pd.DataFrame({
    "Model": ["SVM", "Random Forest", "AdaBoost"],

    "Accuracy Before": [
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada)
    ],

    "Accuracy After": [
        accuracy_score(y_test, y_pred_svm_opt),
        accuracy_score(y_test, y_pred_rf_opt),
        accuracy_score(y_test, y_pred_ada_opt)
    ],

    "Precision Before": [
        precision_score(y_test, y_pred_svm, pos_label="M"),
        precision_score(y_test, y_pred_rf, pos_label="M"),
        precision_score(y_test, y_pred_ada, pos_label="M")
    ],

    "Precision After": [
        precision_score(y_test, y_pred_svm_opt, pos_label="M"),
        precision_score(y_test, y_pred_rf_opt, pos_label="M"),
        precision_score(y_test, y_pred_ada_opt, pos_label="M")
    ],

    "Recall Before": [
        recall_score(y_test, y_pred_svm, pos_label="M"),
        recall_score(y_test, y_pred_rf, pos_label="M"),
        recall_score(y_test, y_pred_ada, pos_label="M")
    ],

    "Recall After": [
        recall_score(y_test, y_pred_svm_opt, pos_label="M"),
        recall_score(y_test, y_pred_rf_opt, pos_label="M"),
        recall_score(y_test, y_pred_ada_opt, pos_label="M")
    ],

    "F1-Score Before": [
        f1_score(y_test, y_pred_svm, pos_label="M"),
        f1_score(y_test, y_pred_rf, pos_label="M"),
        f1_score(y_test, y_pred_ada, pos_label="M")
    ],

    "F1-Score After": [
        f1_score(y_test, y_pred_svm_opt, pos_label="M"),
        f1_score(y_test, y_pred_rf_opt, pos_label="M"),
        f1_score(y_test, y_pred_ada_opt, pos_label="M")
    ]
})

print("Model Performance Comparison: Before vs After Optimization\n")
display(comparison.round(4))

Model Performance Comparison: Before vs After Optimization



,Model,Accuracy Before,Accuracy After,Precision Before,Precision After,Recall Before,Recall After,F1-Score Before,F1-Score After
0,SVM,0.9737,0.9825,1.0,1.0,0.9286,0.9524,0.9630,0.9756
1,Random Forest,0.9737,0.9737,1.0,1.0,0.9286,0.9286,0.9630,0.9630
2,AdaBoost,0.9825,0.9737,1.0,1.0,0.9524,0.9286,0.9756,0.9630


In [42]:
best_model_name = comparison.loc[
    comparison["F1-Score After"].idxmax(),
    "Model"
]

best_model_score = comparison["F1-Score After"].max()

print("Best Performing Model:", best_model_name)
print("Best Test F1-Score:", round(best_model_score, 4))

Best Performing Model: SVM
Best Test F1-Score: 0.9756


In [46]:
df.describe()

,2,3,4,5,6,7,8,9,10,11,...,22,23,24,25,26,27,28,29,30,31
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500
